# Model 06: genealogical MRCA and the Identical Ancestors Point

This notebook traces a complete two-parent pedigree backward from the present generation.

- **MRCA:** first past generation containing at least one ancestor of everyone alive in the present generation.
- **IAP:** first past generation in which every individual with any present-day descendants is an ancestor of everyone in the present generation.

This is genealogical ancestry only. It does not model DNA.

In [ ]:
import os, sys, subprocess
if 'google.colab' in sys.modules:
    if not os.path.exists('/content/Evolution-Creation'):
        subprocess.run(['git','clone','-q','https://github.com/vafaei-ar/Evolution-Creation.git','/content/Evolution-Creation'], check=True)
    subprocess.run([sys.executable,'-m','pip','install','-q','-e','/content/Evolution-Creation'], check=True)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from IPython.display import display
from evolution_creation.coalescence import (
    chang_asymptotic_generations,
    make_parent_source_matrix,
    simulate_coalescence_replicates,
    simulate_pedigree_coalescence,
    simulate_structured_coalescence_replicates,
)


## A. Chang-style random-mating baseline

The exact simulation uses finite pedigrees. The theoretical values printed below are Chang's large-population asymptotic benchmarks, not fitted historical dates.

In [ ]:
population_size=widgets.IntSlider(value=500,min=50,max=3000,step=50,description='N')
max_generations=widgets.IntSlider(value=40,min=10,max=100,step=5,description='Max gen')
replicates=widgets.IntSlider(value=100,min=10,max=300,step=10,description='Replicates')
years_per_generation=widgets.FloatSlider(value=25.0,min=20.0,max=35.0,step=0.5,description='Years/gen')
seed=widgets.IntText(value=20260920,description='Seed')
display(population_size,max_generations,replicates,years_per_generation,seed)

In [ ]:
def run_baseline(_=None):
    n=population_size.value
    exact=simulate_pedigree_coalescence(n,max_generations.value,seed.value,stop_at_iap=False)
    x=exact.generations_back
    fig,ax=plt.subplots(figsize=(10,5))
    ax.plot(x,exact.noncontributing_counts/n,label='no present descendants')
    ax.plot(x,exact.partial_counts/n,label='partial ancestors')
    ax.plot(x,exact.universal_counts/n,label='universal ancestors')
    if exact.mrca_generation is not None: ax.axvline(exact.mrca_generation,linestyle='--',label=f'MRCA = {exact.mrca_generation}')
    if exact.iap_generation is not None: ax.axvline(exact.iap_generation,linestyle=':',label=f'IAP = {exact.iap_generation}')
    ax.set(xlabel='Generations before present',ylabel='Fraction of past generation',ylim=(0,1.02))
    ax.legend(ncol=2); plt.show()

    summary=simulate_coalescence_replicates(n,max_generations.value,replicates.value,seed.value)
    finite_m=summary.mrca_generations[np.isfinite(summary.mrca_generations)]
    finite_i=summary.iap_generations[np.isfinite(summary.iap_generations)]
    fig,ax=plt.subplots(figsize=(9,4))
    bins=np.arange(0,max_generations.value+2)-0.5
    ax.hist(finite_m,bins=bins,alpha=.6,label='MRCA')
    ax.hist(finite_i,bins=bins,alpha=.6,label='IAP')
    ax.set(xlabel='Generations before present',ylabel='Replicates')
    ax.legend(); plt.show()

    bench_m,bench_i=chang_asymptotic_generations(n)
    y=years_per_generation.value
    print(f'Chang asymptotic MRCA benchmark: {bench_m:.2f} generations (~{bench_m*y:.0f} years)')
    print(f'Chang asymptotic IAP benchmark:  {bench_i:.2f} generations (~{bench_i*y:.0f} years)')
    print(f'Simulation median MRCA: {summary.median_mrca_generation:.1f} generations')
    print(f'Simulation median IAP:  {summary.median_iap_generation:.1f} generations')
    print(f'MRCA reached within window: {summary.mrca_reached_fraction:.1%}')
    print(f'IAP reached within window:  {summary.iap_reached_fraction:.1%}')

baseline_button=widgets.Button(description='Run baseline',button_style='primary')
baseline_button.on_click(run_baseline)
display(baseline_button)
run_baseline()

## B. Population structure

Isolation 0 is population-size-proportional panmixia. Isolation 1 forces both parents to come from the child's own community, so a global MRCA is impossible across disconnected communities.

In [ ]:
communities=widgets.IntSlider(value=4,min=2,max=8,description='Communities')
community_size=widgets.IntSlider(value=100,min=20,max=400,step=20,description='Size/group')
isolation=widgets.FloatSlider(value=0.98,min=0.0,max=1.0,step=0.005,readout_format='.3f',description='Isolation')
structured_replicates=widgets.IntSlider(value=40,min=5,max=100,step=5,description='Replicates')
structured_max_generations=widgets.IntSlider(value=100,min=20,max=200,step=10,description='Max gen')
display(communities,community_size,isolation,structured_replicates,structured_max_generations)

In [ ]:
def run_structure(_=None):
    sizes=[community_size.value]*communities.value
    matrix=make_parent_source_matrix(sizes,isolation.value)
    summary=simulate_structured_coalescence_replicates(
        community_sizes=sizes,
        parent_source_matrix=matrix,
        max_generations=structured_max_generations.value,
        replicates=structured_replicates.value,
        seed=seed.value,
    )
    fig,ax=plt.subplots(figsize=(6,5))
    image=ax.imshow(matrix,vmin=0,vmax=1)
    ax.set(xlabel='Parent source community',ylabel='Child community',title='Parent-source matrix')
    fig.colorbar(image,ax=ax,label='Probability'); plt.show()
    print(f'MRCA reached: {summary.mrca_reached_fraction:.1%}')
    print(f'IAP reached:  {summary.iap_reached_fraction:.1%}')
    print(f'Median MRCA among reached runs: {summary.median_mrca_generation:.1f}')
    print(f'Median IAP among reached runs:  {summary.median_iap_generation:.1f}')

structure_button=widgets.Button(description='Run structured model',button_style='primary')
structure_button.on_click(run_structure)
display(structure_button)
run_structure()

## Interpretation

A recent genealogical MRCA in a random-mating model is a mathematical result about that model, not a historical date. Population structure can delay coalescence substantially, and complete persistent disconnection prevents a global MRCA altogether. Conversely, any nonzero reproductive bridge can eventually connect pedigrees, depending on the available time.